In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.utils import shuffle
from scipy.stats import uniform

In [ ]:
# 가상의 데이터셋 생성 (CIFAR-10과 유사한 데이터셋)
num_classes = 10
input_shape = (32, 32, 3)
num_samples = 1000

X = np.random.rand(num_samples, *input_shape)
y = np.random.randint(num_classes, size=num_samples)

In [ ]:
# 데이터셋 셔플 및 분할
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
def create_model(learning_rate=0.001):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    opt = Adam(learning_rate=learning_rate)

    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# 하이퍼파라미터 분포 정의
param_dist = {
    'learning_rate': uniform(0.001, 0.1),
    'batch_size': [16, 32, 64]
}

best_accuracy = 0
best_params = {}
n_iter = 5

# 랜덤하게 하이퍼파라미터 조합을 선택하여 시도
for params in ParameterSampler(param_dist, n_iter=n_iter, random_state=42):
    model = create_model(learning_rate=params['learning_rate'])
    model.fit(X_train, y_train, epochs=10, batch_size=params['batch_size'], validation_split=0.2, verbose=0)

    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"Params: {params}, Test Accuracy: {accuracy}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = params

print("Best Parameters (RandomSearch):", best_params)
print("Best Test Accuracy (RandomSearch):", best_accuracy)

In [ ]:
# 최적의 하이퍼파라미터로 모델 재학습
best_model = create_model(learning_rate=best_params['learning_rate'])
best_model.fit(X_train, y_train, epochs=10, batch_size=best_params['batch_size'], validation_split=0.2)

# 테스트 데이터셋으로 모델 평가
test_loss, test_accuracy = best_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2f}")
print(f"Test Loss: {test_loss:.2f}")